In [1]:
from dotenv import load_dotenv    
import os                         

load_dotenv()                    

from langchain_openai import ChatOpenAI                          
from langchain_core.messages import HumanMessage, SystemMessage  
from langgraph.graph import StateGraph, START, END               
from langgraph.graph.message import add_messages                 
from typing import TypedDict, Annotated                          


class State(TypedDict):
    messages: Annotated[list, add_messages]   

In [2]:

llm = ChatOpenAI(
    model="gpt-4o-mini",                              
    api_key=os.getenv("API_TOKEN"),                    
    base_url="https://openrouter.ai/api/v1"            
)


def chatbot(state: State) -> dict:
    """
    The chatbot node. Takes the current messages,
    sends them to the LLM, and returns the response.
    """
    
    system = SystemMessage(content="You are a helpful and friendly assistant.")
    
    
    response = llm.invoke([system] + state["messages"])
    
    
    return {"messages": [response]}

In [3]:

graph_builder = StateGraph(State)              

graph_builder.add_node("chatbot", chatbot)     

graph_builder.add_edge(START, "chatbot")       
graph_builder.add_edge("chatbot", END)        

graph = graph_builder.compile()

In [4]:

result = graph.invoke({
    "messages": [HumanMessage(content="What are 3 fun facts about the ocean?")]
})


for msg in result["messages"]:
    if hasattr(msg, 'content'):
        
        role = "Human" if isinstance(msg, HumanMessage) else "AI"
        print(f"{role}: {msg.content}\n")

Human: What are 3 fun facts about the ocean?

AI: Sure! Here are three fun facts about the ocean:

1. **Vast Depths**: The ocean covers over 70% of the Earth's surface and is home to the Mariana Trench, the deepest part of the world's oceans, which reaches about 36,000 feet (around 11,000 meters) below sea level. That’s deeper than Mount Everest is tall!

2. **Bioluminescence**: Many ocean creatures can produce their own light through a process called bioluminescence. This phenomenon is utilized for various purposes, such as attracting prey, communication, and camouflage. Organisms like certain jellyfish and the lanternfish are famous examples of bioluminescent species.

3. **Oxygen Production**: The ocean is responsible for producing about 50-80% of the world’s oxygen! Phytoplankton, tiny ocean plants, carry out photosynthesis and contribute significantly to oxygen levels in the atmosphere.

The ocean is an incredibly diverse and essential part of our planet!

